# AlphaPose Experimentation Notebook

This notebook allows you to experiment with AlphaPose for Human Pose Estimation directly in Google Colab.
It adapts the `main.py` script for interactive use.

**Prerequisites:**
- Ensure you have uploaded your input video/image files or mounted your Google Drive.
- The AlphaPose model files (`models/AlphaPose/`) and other necessary Python modules (`movenet_hpe.py`, `openvino_base_hpe.py`, `base_hpe.py`) should be accessible. You might need to upload them or clone the repository.


In [ ]:
# Install necessary libraries
!pip install opencv-python numpy torch torchvision -q

# If AlphaPose modules are not in the current Colab environment, you might need to clone the repository or upload them.
# For example, if you have the project structure locally, you can upload the 'models' and other relevant .py files.
# For demonstration, we'll assume the files are available in the Colab environment.
# If you need to clone a repo:
# !git clone <repository_url>
# And adjust sys.path accordingly.

# Add the current directory to sys.path to find local modules
import sys
import os
sys.path.insert(0, os.getcwd())

# For demonstration, let's assume 'models/AlphaPose' and other .py files are in the current directory or accessible.
# If not, you'll need to upload them or adjust paths.


In [ ]:
import cv2
import numpy as np
import torch
import logging
import argparse # Keep for structure, but will be replaced by hardcoded values

# Import custom modules
# Ensure these files are accessible in your Colab environment
try:
    from movenet_hpe import MoveNetHPE
    from openvino_base_hpe import OpenVINOBaseHPE
    from alphapose_hpe import AlphaPoseHPE
    from base_hpe import BaseHPE, Body, Padding # Import BaseHPE components if needed for custom logic
except ImportError as e:
    print(f"Error importing custom modules: {e}")
    print("Please ensure 'movenet_hpe.py', 'openvino_base_hpe.py', 'alphapose_hpe.py', and 'base_hpe.py' are in your Colab environment.")
    # You might need to upload these files or clone the repository.

# Configure logging
class _HideAspectWarn(logging.Filter):
    def filter(self, rec):
        msg = rec.getMessage()
        return "Chosen model aspect ratio doesn't match image aspect ratio" not in msg

logging.getLogger().addFilter(_HideAspectWarn())

# Helper function to prepare arguments (adapted from main.py)
def get_hpe_args(method, input_src, output_dir=None, enable_json=False, enable_csv=False,
                 measurement_interval_ms=100, save_image=False, save_video=False,
                 device="GPU", detbatch=5, posebatch=4): # Added posebatch here for clarity
    
    # Create a simple object to mimic argparse.Namespace
    class Args:
        def __init__(self, **kwargs):
            self.__dict__.update(kwargs)
    
    args = Args(
        method=method,
        input=input_src,
        output_dir=output_dir,
        json=enable_json,
        csv=enable_csv,
        measurement_interval_ms=measurement_interval_ms,
        save_video=save_video,
        save_image=save_image,
        device=device,
        detbatch=detbatch
    )
    return args

# Helper function to prepare base arguments (adapted from main.py)
def base_args(args):
    return {
        "input_src": args.input,
        "output_dir": args.output_dir,
        "enable_json": args.json,
        "enable_csv": args.csv,
        "measurement_interval_ms": args.measurement_interval_ms,
        "save_image": args.save_image,
        "save_video": args.save_video
    }

# Function to get the HPE method (adapted from main.py)
def get_hpe_method(args):
    method_map = {
        'movenet': lambda args: MoveNetHPE(device=args.device, **base_args(args)),
        'alphapose': lambda args: AlphaPoseHPE(device=args.device, detbatch=args.detbatch, **base_args(args)),
        'openpose': lambda args: OpenVINOBaseHPE(model_type='openpose', device=args.device, **base_args(args)),
        'hrnet': lambda args: OpenVINOBaseHPE(model_type='higherhrnet', device=args.device, **base_args(args)),
        'ae1': lambda args: OpenVINOBaseHPE(model_type='efficienthrnet1', device=args.device, **base_args(args)),
        'ae2': lambda args: OpenVINOBaseHPE(model_type='efficienthrnet2', device=args.device, **base_args(args)),
        'ae3': lambda args: OpenVINOBaseHPE(model_type='efficienthrnet3', device=args.device, **base_args(args)),
    }

    name = args.method.lower()

    if name not in method_map:
        raise ValueError(f"Unknown method: {name}")

    if callable(method_map[name]):
        return method_map[name](args)
    else:
        return method_map[name](**base_args(args))


In [ ]:
# --- Configuration ---
# Set your desired parameters here
METHOD = 'alphapose' # e.g., 'alphapose', 'movenet', 'openpose'
# IMPORTANT: For Colab, use paths accessible in the environment.
# Upload files directly, or mount Google Drive and use paths like '/content/drive/MyDrive/videos/my_video.mp4'
INPUT_SOURCE = 'path/to/your/video.mp4' # e.g., '0' for webcam (if available), 'path/to/video.mp4', 'path/to/image.jpg'
DEVICE = 'CPU' # 'GPU' or 'CPU'. For Colab, 'GPU' might be available if you select a GPU runtime.
DETBATCH_SIZE = 4 # Detection batch size
# POSEBATCH_SIZE is hardcoded to 4 in alphapose_hpe.py and not exposed via args.
# If you need to change it, modify alphapose_hpe.py directly.
OUTPUT_DIR = 'output_results'
ENABLE_JSON = False
ENABLE_CSV = False
SAVE_IMAGE = False
SAVE_VIDEO = False

# --- Execution ---
print(f"Running {METHOD} with device: {DEVICE}, detbatch: {DETBATCH_SIZE}")

# Prepare arguments
args = get_hpe_args(
    method=METHOD,
    input_src=INPUT_SOURCE,
    output_dir=OUTPUT_DIR,
    enable_json=ENABLE_JSON,
    enable_csv=ENABLE_CSV,
    save_image=SAVE_IMAGE,
    save_video=SAVE_VIDEO,
    device=DEVICE,
    detbatch=DETBATCH_SIZE
)

# Instantiate and run HPE
try:
    hpe_instance = get_hpe_method(args)
    hpe_instance.load_model()
    hpe_instance.main_loop()
    print("Processing complete.")
except Exception as e:
    print(f"An error occurred during execution: {e}")
    import traceback
    traceback.print_exc()


## How to Use

1.  **Install Dependencies**: Run the first code cell to install required libraries.
2.  **Upload Files**: 
    *   Upload your input video or image files to Colab or mount your Google Drive.
    *   Ensure the AlphaPose model files (`models/AlphaPose/`) and other necessary Python scripts (`movenet_hpe.py`, `openvino_base_hpe.py`, `base_hpe.py`, `alphapose_hpe.py`) are accessible in your Colab environment. You might need to upload them or clone the repository.
3.  **Configure Parameters**: In the second-to-last code cell, modify the `METHOD`, `INPUT_SOURCE`, `DEVICE`, `DETBATCH_SIZE`, and other parameters as needed.
    *   For `INPUT_SOURCE`, use the path to your uploaded file or a file in your mounted Google Drive (e.g., `/content/drive/MyDrive/videos/my_video.mp4`).
    *   Set `DEVICE` to 'CPU' or 'GPU' (if available and configured).
    *   Adjust `DETBATCH_SIZE` for performance tuning.
4.  **Run the Notebook**: Execute the remaining code cells.
